
# PDF QA Sistemi

## Amaç
Bu projede PDF dosyaları üzerinden soru-cevap sistemi geliştirilmiştir.

Sistem:
- PDF dosyasını okur
- Metni işler
- Kullanıcının sorduğu soruya PDF içeriğine göre cevap verir

## Kullanılan Teknolojiler
- Python
- LangChain
- FAISS
- Sentence Transformers
- Transformers



# 1. Gerekli Kütüphanelerin Kurulması


In [7]:

!pip install PyPDF2
!pip install langchain
!pip install sentence-transformers
!pip install faiss-cpu
!pip install transformers
!pip install torch

import PyPDF2
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 21.2 MB/s eta 0:00:00



# 2. PDF Dosyasının Yüklenmesi

Bu aşamada PDF dosyasının içeriği okunur.


In [8]:
from PyPDF2 import PdfReader

# pdf_path = "Elektronik2malzeme.pdf"   # Buraya kendi PDF dosyanızın adını yazın
# You need to upload your PDF file (e.g., Elektronik2malzeme.pdf) to your Colab environment.
# For example, you can drag and drop it into the files section on the left sidebar.
# Once uploaded, you can uncomment the line above or specify the correct path if it's in a subdirectory.
# For now, let's assume you'll upload 'Elektronik2malzeme.pdf' to the root directory.

pdf_path = open("emalzeme.pdf", "rb")  # Placeholder for the user's PDF file

reader = PyPDF2.PdfReader(pdf_path)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print(text[:2000])

Elektronik II Dersi Laboratuvar Malzeme Listesi 
- Breadboard 
- 18 kΩ, 4 x 10kΩ, 5 kΩ, 4.7 kΩ, 3.3 kΩ, 2.2 kΩ, 2 x 1 kΩ, 220 Ω Direnç 
- 50 kΩ Potansiyometre 
- 47 µF Kapasitör 
- 2 x LM741 
- 2 x 1N4001 
- BT151 
- 2 x Switch 
- 2 x LED 



# 3. Metnin Parçalara Ayrılması

Uzun metinler daha iyi işlenebilmesi için küçük parçalara bölünür.


In [12]:
chunks = text.split(".")

print("Toplam parça sayısı:" , len(chunks))

Toplam parça sayısı: 4



# 4. Embedding Oluşturma

Metin parçaları sayısal vektörlere dönüştürülür.


In [13]:

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embedding_model.encode(chunks)

print("Embedding Boyutu:", embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Boyutu: (4, 384)



# 5. FAISS Veritabanı Oluşturma

Benzer metinleri hızlı bulabilmek için FAISS kullanılır.


In [14]:

import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("FAISS veritabanına eklenen parça sayısı:", index.ntotal)


FAISS veritabanına eklenen parça sayısı: 4



# 6. Soru-Cevap Sistemi

Kullanıcının sorusuna en uygun PDF parçası bulunur ve cevap üretilir.


In [24]:

from transformers import pipeline

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

In [27]:

question = input("Sorunuzu girin: ")

question_embedding = embedding_model.encode([question])

k = 1

distances, indices = index.search(
    np.array(question_embedding),
    k
)

relevant_text = chunks[indices[0][0]]

prompt = f'''
Aşağıdaki metne göre soruyu cevapla.

Metin:
{relevant_text}

Soru:
{question}
'''

result = qa_pipeline(
    prompt,
    max_length=100
)

print("\nCevap:")
print(result[0]['generated_text'])


Sorunuzu girin: kaç tane led gerekiyor


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Cevap:

Aşağıdaki metne göre soruyu cevapla.

Metin:
2 kΩ, 2 x 1 kΩ, 220 Ω Direnç 
- 50 kΩ Potansiyometre 
- 47 µF Kapasitör 
- 2 x LM741 
- 2 x 1N4001 
- BT151 
- 2 x Switch 
- 2 x LED 

Soru:
kaç tane led gerekiyor
ruhan-eit. I say" The main culprit, though some experts agree



# 7. Sonuç

Bu projede PDF dosyaları üzerinde çalışan bir soru-cevap sistemi geliştirilmiştir.

Sistem:
- PDF içeriğini okuyabilmektedir
- Metni işleyebilmektedir
- Kullanıcı sorularına uygun cevap verebilmektedir

Bu yöntem özellikle:
- Doküman analizi
- Akademik araştırmalar
- Rapor inceleme
- Bilgi erişim sistemleri

gibi alanlarda kullanılabilir.
